In [1]:
#importing required libraries
from embedder import Embedder
import numpy as np
from gitsource import GithubRepositoryDataReader
from gitsource import chunk_documents
from minsearch import VectorSearch
from minsearch import Index

2026-06-29 12:34:08.463822697 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [2]:
#creating an instance of the Embedder class
embed = Embedder()

In [3]:
#encoding the question using the embedder instance
q1 = "How does approximate nearest neighbor search work?"
q1_embed = embed.encode(q1)

In [4]:
#viewing the first element of the embedded question
q1_embed[0]
#Q1 answer:

np.float64(-0.02058203437252893)

In [5]:
# reading the repository data using the GithubRepositoryDataReader class and filtering for markdown files in the lessons directory and parsing them into documents


reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [6]:
# viewing the first document and its length
print(documents[0])
len(documents)

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

72

In [7]:
# creating a list of content from the documents where the filename matches a specific lesson and encoding that content using the embedder instance
content = [ doc["content"] for doc in documents if doc["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md" ]
content_embed = embed.encode(content[0])

In [8]:
# computing the dot product(Cosine similarity) between the embedded question and the embedded content
q1_embed.dot(content_embed)
#Q2 answer:

np.float64(0.36107027225589694)

In [9]:
# chunking the documents into smaller pieces for processing
chunks = chunk_documents(documents, size=2000, step=1000)

In [10]:
# encoding the chunks using the embedder instance
chunk_embeds = embed.encode_batch([chunk["content"] for chunk in chunks])

In [11]:
# creating a numpy array from the chunk embeddings for further processing
X = np.array(chunk_embeds)

In [12]:
# viewing the shape of the numpy array containing the chunk embeddings
X.shape

(295, 384)

In [13]:
# computing the dot product between the embedded question and the chunk embeddings to find the most relevant chunk 
scores = X.dot(q1_embed)
id = np.argmax(scores)
#print(chunks[id])
print(chunks[id]["filename"])
# Q3 answer:

02-vector-search/lessons/07-sqlitesearch-vector.md


In [14]:
# scores = chunk_embeds.dot(q1_embed)
# id = np.argmax(scores)
# id,scores[id]
# chunks[id]
# chunks[id]["filename"]

In [15]:
# creating a new question and encoding it using the embedder instance
q4 = "What metric do we use to evaluate a search engine?"
q4_embed = embed.encode(q4)

In [16]:
# creating a vector search index and fitting it with the chunk embeddings
vindex = VectorSearch()
vindex.fit(X, chunks)

In [17]:
# searching the vector index for the most relevant chunks to the Q4 question
results = vindex.search(q4_embed, num_results=5)

In [18]:
# viewing the filename of the most relevant chunk to the Q4 question
results[0]["filename"]
#Q4 answer:

'04-evaluation/lessons/05-search-metrics.md'

In [19]:
# creating a new question and encoding it using the embedder instance
q5 = "How do I store vectors in PostgreSQL?"
q5_embed = embed.encode(q5)


In [20]:
# creating a new index for searching the chunks based on their content and fitting it with the chunk data
# q5_kws is the lexical search results for the Q5 question, with a boost applied to the content field to prioritize matches in that field
index = Index(
    text_fields=["content"]
)
index.fit(chunks)
q5_kws = index.search(q5,
             boost_dict={"content": 3.0},
             num_results=5

)

In [21]:
# viewing the filenames of the most relevant chunks to the Q5 question by lexical search
kws_result = [doc["filename"] for doc in q5_kws]
kws_result

['02-vector-search/lessons/02-embeddings.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md']

In [22]:
# searching the vector index for the most relevant chunks to the Q5 question
q5_results = vindex.search(q5_embed, num_results=5)

In [23]:
# viewing the filenames of the most relevant chunks to the Q5 question by searching the vector index
vec_result = [doc["filename"] for doc in q5_results]
vec_result

['02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md']

In [24]:
print(vec_result)

['02-vector-search/lessons/08-pgvector.md', '02-vector-search/lessons/08-pgvector.md', '03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/08-pgvector.md', '02-vector-search/lessons/08-pgvector.md']


In [25]:
# listing the filenames that are present in the vector search results but not in the text search results for the Q5 question
list(set(vec_result) - set(kws_result))
#Q5 answer:

['02-vector-search/lessons/08-pgvector.md']

In [26]:
# Reciprocal Rank Fusion 
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [27]:
# creating a new question and encoding it using the embedder instance
q6 = "How do I give the model access to tools?"
q6_embed = embed.encode(q6)

In [28]:
# searching the vector index for the most relevant chunks to the Q6 question
q6_results = vindex.search(q6_embed)

In [29]:
# searching the index for the most relevant chunks to the Q6 question by lexical search, with a boost applied to the content field to prioritize matches in that field
q6_kws = index.search(q6,
             boost_dict={"content": 2.0}
)

In [30]:
# combining the results from both the vector search and lexical search for the Q6 question using Reciprocal Rank Fusion
q6_results = rrf([q6_results, q6_kws])


In [31]:
# viewing the filename of the most relevant chunk to the Q6 question
q6_results[0]["filename"]
#Q6 answer:

'01-agentic-rag/lessons/13-function-calling.md'